In [1]:
import os
import time
import numpy as np
import pandas as pd
import tensorflow as tf

from itertools import product
from sklearn.model_selection import train_test_split

from tensorflow.keras.applications import EfficientNetB0, ResNet50, MobileNetV3Small
from tensorflow.keras.optimizers import Adam


2025-08-26 09:31:50.644811: I tensorflow/core/util/port.cc:113] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-26 09:31:50.672600: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2025-08-26 09:31:50.811704: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-08-26 09:31:50.811770: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-08-26 09:31:50.837912: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
INITIAL_EPOCHS = 15
FINE_TUNING_EPOCHS = 10

DATA_DIR = "../datasets/fondo_blanco"
SEED = 42

tf.random.set_seed(SEED)
np.random.seed(SEED)

print("📊 Configuración")
print(f" - IMG_SIZE: {IMG_SIZE}")
print(f" - BATCH_SIZE: {BATCH_SIZE}")
print(f" - EPOCHS Fase1: {INITIAL_EPOCHS}")
print(f" - EPOCHS Fase2: {FINE_TUNING_EPOCHS}")
print(f" - DATA_DIR: {DATA_DIR}")


📊 Configuración
 - IMG_SIZE: (224, 224)
 - BATCH_SIZE: 32
 - EPOCHS Fase1: 15
 - EPOCHS Fase2: 10
 - DATA_DIR: ../datasets/fondo_blanco


In [3]:
def load_and_preprocess_data(data_dir, img_size=IMG_SIZE):
    if not os.path.exists(data_dir):
        raise ValueError(f"El directorio {data_dir} no existe")
    classes = [d for d in sorted(os.listdir(data_dir)) if os.path.isdir(os.path.join(data_dir, d))]
    if not classes:
        raise ValueError(f"No hay subdirectorios (clases) en {data_dir}")
    print(f"🔍 Clases: {classes}")

    images, labels = [], []
    for idx, class_name in enumerate(classes):
        class_dir = os.path.join(data_dir, class_name)
        image_files = [f for f in os.listdir(class_dir) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
        print(f"   📁 '{class_name}': {len(image_files)} imágenes")
        for img_name in image_files:
            path = os.path.join(class_dir, img_name)
            try:
                img = tf.keras.preprocessing.image.load_img(path, target_size=img_size)
                arr = tf.keras.preprocessing.image.img_to_array(img)
                images.append(arr)
                labels.append(idx)
            except Exception as e:
                print(f"   ⚠️ Error en {path}: {e}")

    if not images:
        raise ValueError("No se pudieron cargar imágenes")

    X = np.array(images, dtype=np.float32)
    y = tf.keras.utils.to_categorical(np.array(labels), num_classes=len(classes))

    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.20, random_state=SEED, stratify=y)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.20, random_state=SEED, stratify=y_temp)

    print(f"✅ Total: {len(X)} | Train: {X_train.shape[0]} | Val: {X_val.shape[0]} | Test: {X_test.shape[0]}")
    return X_train, X_val, X_test, y_train, y_val, y_test, classes

def create_data_augmentation():
    aug_layers = [
        tf.keras.layers.RandomFlip("horizontal"),
        tf.keras.layers.RandomRotation(0.1),
        tf.keras.layers.RandomTranslation(0.05, 0.05),
        tf.keras.layers.RandomZoom(0.1),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.GaussianNoise(0.05),
    ]
    # Fallback para RandomBrightness (no siempre está):
    if hasattr(tf.keras.layers, "RandomBrightness"):
        aug_layers.insert(5, tf.keras.layers.RandomBrightness(0.2))
    else:
        def random_brightness(x):
            delta = tf.random.uniform([], -0.2, 0.2)
            return tf.image.adjust_brightness(x, delta)
        aug_layers.insert(5, tf.keras.layers.Lambda(random_brightness))
    return tf.keras.Sequential(aug_layers)

def create_datasets(X_train, X_val, X_test, y_train, y_val, y_test, batch_size, preprocess_func):
    AUTOTUNE = tf.data.AUTOTUNE
    aug = create_data_augmentation()

    def map_train(image, label):
        image = tf.cast(image, tf.float32)
        image = aug(image, training=True)
        image = preprocess_func(image)
        return image, label

    def map_eval(image, label):
        image = tf.cast(image, tf.float32)
        image = preprocess_func(image)
        return image, label

    shuffle_buf = min(512, len(X_train))  # menor buffer -> más rápido en CPU
    train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
    val_ds   = tf.data.Dataset.from_tensor_slices((X_val, y_val))
    test_ds  = tf.data.Dataset.from_tensor_slices((X_test, y_test))

    train_ds = (train_ds
                .cache()
                .shuffle(shuffle_buf, seed=SEED)
                .map(map_train, num_parallel_calls=AUTOTUNE)
                .batch(batch_size)
                .prefetch(AUTOTUNE))

    val_ds = (val_ds
              .cache()
              .map(map_eval, num_parallel_calls=AUTOTUNE)
              .batch(batch_size)
              .prefetch(AUTOTUNE))

    test_ds = (test_ds
               .cache()
               .map(map_eval, num_parallel_calls=AUTOTUNE)
               .batch(batch_size)
               .prefetch(AUTOTUNE))

    return train_ds, val_ds, test_ds


In [ ]:
def build_head(num_classes, units1=768, units2=384, drop1=0.5, drop2=0.4):
    return tf.keras.Sequential([
        tf.keras.layers.GlobalAveragePooling2D(),
        tf.keras.layers.Dense(units1, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.Dropout(drop1),
        tf.keras.layers.Dense(units2, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(1e-4)),
        tf.keras.layers.Dropout(drop2),
        tf.keras.layers.Dense(num_classes, activation='softmax')
    ], name="head")

def create_model_with_params(arch_name, input_shape, num_classes, base_weights, head_cfg):
    if arch_name == "ResNet50":
        base_model = ResNet50(input_shape=input_shape, include_top=False, weights=base_weights)
    elif arch_name == "MobileNetV3":
        base_model = MobileNetV3Small(input_shape=input_shape, include_top=False, weights=base_weights)
    else:
        raise ValueError("Arquitectura no soportada (usa 'ResNet50' o 'MobileNetV3').")

    base_model.trainable = False
    head = build_head(num_classes, **head_cfg)
    model = tf.keras.Sequential([base_model, head])
    return model, base_model

def compile_model(model, lr, label_smoothing=0.0):
    loss = tf.keras.losses.CategoricalCrossentropy(label_smoothing=label_smoothing)
    model.compile(optimizer=Adam(learning_rate=lr), loss=loss, metrics=['accuracy'])

def callbacks_common():
    return [
        tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', patience=2, restore_best_weights=True),
        tf.keras.callbacks.ReduceLROnPlateau(monitor='val_accuracy', factor=0.3, patience=1, min_lr=1e-6)
    ]

In [ ]:
GRID_FASE1 = {
    "ResNet50": {
        "base_weights": ["imagenet"],
        "units1": [768],
        "units2": [384],
        "drop1": [0.5],
        "drop2": [0.3, 0.4],
        "lr_init": [3e-4, 1e-3]
    },
    "MobileNetV3": {
        "base_weights": ["imagenet"],
        "units1": [512],
        "units2": [256],
        "drop1": [0.4],
        "drop2": [0.3],
        "lr_init": [3e-4, 1e-3]
    }
}

GRID_FASE2 = {
    "ResNet50": {
        "unfreeze_last_n": [30],
        "lr_fine": [3e-5, 1e-4]
    },
    "MobileNetV3": {
        "unfreeze_last_n": [40],
        "lr_fine": [3e-5, 1e-4]
    }
}

In [ ]:
# =======================
# Grid Search
# =======================
def grid_search_fase1(arch_name, input_shape, num_classes, train_ds, val_ds, max_epochs=INITIAL_EPOCHS):
    grid = GRID_FASE1[arch_name]
    keys = ["base_weights", "units1", "units2", "drop1", "drop2", "lr_init"]
    best = None
    trials = []

    for vals in product(*[grid[k] for k in keys]):
        cfg = dict(zip(keys, vals))
        head_cfg = {"units1": cfg["units1"], "units2": cfg["units2"], "drop1": cfg["drop1"], "drop2": cfg["drop2"]}

        model, base = create_model_with_params(arch_name, input_shape, num_classes,
                                               base_weights=cfg["base_weights"], head_cfg=head_cfg)
        compile_model(model, cfg["lr_init"], label_smoothing=0.05)
        hist = model.fit(train_ds, validation_data=val_ds,
                         epochs=max_epochs, callbacks=callbacks_common(), verbose=0)

        val_acc = max(hist.history["val_accuracy"])
        trials.append({**cfg, "val_acc": float(val_acc)})
        if (best is None) or (val_acc > best["val_acc"]):
            best = {"cfg": cfg, "val_acc": float(val_acc), "weights": model.get_weights()}
        del model, base
        tf.keras.backend.clear_session()

    return best, trials

def apply_unfreeze_last_n(base_model, last_n):
    base_model.trainable = True
    if last_n <= 0:
        return
    for layer in base_model.layers[:-last_n]:
        layer.trainable = False

def rebuild_from_cfg_and_weights(arch_name, input_shape, num_classes, best_cfg_fase1, weights):
    head_cfg = {"units1": best_cfg_fase1["units1"], "units2": best_cfg_fase1["units2"],
                "drop1": best_cfg_fase1["drop1"], "drop2": best_cfg_fase1["drop2"]}
    model, base = create_model_with_params(arch_name, input_shape, num_classes,
                                           base_weights=best_cfg_fase1["base_weights"], head_cfg=head_cfg)
    model.set_weights(weights)
    return model, base

def grid_search_fase2(arch_name, input_shape, num_classes, train_ds, val_ds,
                      best_cfg_fase1, weights_fase1, max_epochs=INITIAL_EPOCHS + FINE_TUNING_EPOCHS):
    grid = GRID_FASE2[arch_name]
    keys = ["unfreeze_last_n", "lr_fine"]
    best = None
    trials = []

    for vals in product(*[grid[k] for k in keys]):
        cfg_ft = dict(zip(keys, vals))

        model, base = rebuild_from_cfg_and_weights(arch_name, input_shape, num_classes, best_cfg_fase1, weights_fase1)
        apply_unfreeze_last_n(base, cfg_ft["unfreeze_last_n"])
        compile_model(model, cfg_ft["lr_fine"], label_smoothing=0.05)

        hist = model.fit(train_ds, validation_data=val_ds,
                         epochs=max_epochs, initial_epoch=INITIAL_EPOCHS,
                         callbacks=callbacks_common(), verbose=0)

        val_acc = max(hist.history["val_accuracy"])
        trials.append({**cfg_ft, "val_acc": float(val_acc)})
        if (best is None) or (val_acc > best["val_acc"]):
            best = {"cfg_ft": cfg_ft, "val_acc": float(val_acc), "weights": model.get_weights()}
        del model, base
        tf.keras.backend.clear_session()

    return best, trials


In [ ]:
def run_architecture_gridsearch(arch, input_shape, num_classes, X_train, X_val, X_test, y_train, y_val, y_test):
    train_ds, val_ds, test_ds = create_datasets(
        X_train, X_val, X_test, y_train, y_val, y_test,
        BATCH_SIZE, arch['preprocess_func']
    )
    arch_name = arch['name']
    print(f"\n=== GRID SEARCH {arch_name}: Fase 1 ===")
    best1, trials1 = grid_search_fase1(arch_name, input_shape, num_classes, train_ds, val_ds, max_epochs=INITIAL_EPOCHS)
    print(f"Mejor Fase 1 {arch_name}: {best1['cfg']} | val_acc={best1['val_acc']:.4f}")

    print(f"\n=== GRID SEARCH {arch_name}: Fase 2 (fine-tuning) ===")
    best2, trials2 = grid_search_fase2(arch_name, input_shape, num_classes, train_ds, val_ds,
                                       best1["cfg"], best1["weights"],
                                       max_epochs=INITIAL_EPOCHS + FINE_TUNING_EPOCHS)
    print(f"Mejor Fase 2 {arch_name}: {best2['cfg_ft']} | val_acc={best2['val_acc']:.4f}")

    final_model, base = rebuild_from_cfg_and_weights(arch_name, input_shape, num_classes, best1["cfg"], best2["weights"])
    compile_model(final_model, 1e-6)
    test_loss, test_acc = final_model.evaluate(test_ds, verbose=0)

    final_name = f"best_{arch_name.lower()}_gridsearch.h5"
    final_model.save(final_name)
    print(f"💾 Guardado: {final_name} | Test Acc: {test_acc:.4f}")

    return {
        "arch": arch_name,
        "best_fase1_cfg": best1["cfg"],
        "best_fase1_val_acc": best1["val_acc"],
        "best_fase2_cfg": best2["cfg_ft"],
        "best_fase2_val_acc": best2["val_acc"],
        "test_acc": float(test_acc),
        "model_path": final_name,
        "trials_fase1": trials1,
        "trials_fase2": trials2,
    }

In [8]:
X_train, X_val, X_test, y_train, y_val, y_test, classes = load_and_preprocess_data(DATA_DIR, IMG_SIZE)
input_shape = (IMG_SIZE[0], IMG_SIZE[1], 3)
num_classes = len(classes)

architectures = [
    {'name': 'ResNet50',   'preprocess_func': tf.keras.applications.resnet.preprocess_input},
    {'name': 'MobileNetV3','preprocess_func': tf.keras.applications.mobilenet_v3.preprocess_input},
]

summary_rows, all_results = [], []
t0 = time.time()
for arch in architectures:
    res = run_architecture_gridsearch(
        arch, input_shape, num_classes, X_train, X_val, X_test, y_train, y_val, y_test
    )
    all_results.append(res)
    summary_rows.append({
        "Arquitectura": res["arch"],
        "Best Fase1": res["best_fase1_cfg"],
        "ValAcc Fase1": round(res["best_fase1_val_acc"], 4),
        "Best Fase2": res["best_fase2_cfg"],
        "ValAcc Fase2": round(res["best_fase2_val_acc"], 4),
        "TestAcc": round(res["test_acc"], 4),
        "Modelo": res["model_path"]
    })

dt = time.time() - t0
df_summary = pd.DataFrame(summary_rows)
print("\n=== RESUMEN GRID SEARCH ===")
print(df_summary.to_string(index=False))
df_summary.to_csv("gridsearch_summary_resnet_mobilenet.csv", index=False)
print(f"🗂️ Resumen guardado en gridsearch_summary_resnet_mobilenet.csv | Tiempo total: {dt:.1f}s")


🔍 Clases: ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U']
   📁 'A': 102 imágenes
   📁 'B': 95 imágenes
   📁 'C': 98 imágenes
   📁 'D': 102 imágenes
   📁 'E': 103 imágenes
   📁 'F': 105 imágenes
   📁 'G': 108 imágenes
   📁 'I': 113 imágenes
   📁 'K': 108 imágenes
   📁 'L': 112 imágenes
   📁 'M': 114 imágenes
   📁 'N': 111 imágenes
   📁 'O': 97 imágenes
   📁 'P': 110 imágenes
   📁 'Q': 120 imágenes
   📁 'R': 90 imágenes
   📁 'S': 100 imágenes
   📁 'T': 102 imágenes
   📁 'U': 108 imágenes
✅ Total: 1998 | Train: 1278 | Val: 320 | Test: 400


2025-08-26 09:33:24.117384: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:901] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2025-08-26 09:33:24.119687: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2256] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...



=== GRID SEARCH ResNet50: Fase 1 ===
Mejor Fase 1 ResNet50: {'base_weights': 'imagenet', 'units1': 768, 'units2': 384, 'drop1': 0.5, 'drop2': 0.4, 'lr_init': 0.001} | val_acc=0.9187

=== GRID SEARCH ResNet50: Fase 2 (fine-tuning) ===
Mejor Fase 2 ResNet50: {'unfreeze_last_n': 30, 'lr_fine': 0.0001} | val_acc=0.9969


/home/juansmc/Documents/uni/AI2/Real-Time-Spanish-Sign-Language-Recognition/.venv/lib/python3.10/site-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


💾 Guardado: best_resnet50_gridsearch.h5 | Test Acc: 0.9800

=== GRID SEARCH MobileNetV3: Fase 1 ===
Mejor Fase 1 MobileNetV3: {'base_weights': 'imagenet', 'units1': 512, 'units2': 256, 'drop1': 0.4, 'drop2': 0.3, 'lr_init': 0.001} | val_acc=0.9375

=== GRID SEARCH MobileNetV3: Fase 2 (fine-tuning) ===
Mejor Fase 2 MobileNetV3: {'unfreeze_last_n': 40, 'lr_fine': 0.0001} | val_acc=0.9688
💾 Guardado: best_mobilenetv3_gridsearch.h5 | Test Acc: 0.9675

=== RESUMEN GRID SEARCH ===
Arquitectura                                                                                               Best Fase1  ValAcc Fase1                                 Best Fase2  ValAcc Fase2  TestAcc                         Modelo
    ResNet50 {'base_weights': 'imagenet', 'units1': 768, 'units2': 384, 'drop1': 0.5, 'drop2': 0.4, 'lr_init': 0.001}        0.9187 {'unfreeze_last_n': 30, 'lr_fine': 0.0001}        0.9969   0.9800    best_resnet50_gridsearch.h5
 MobileNetV3 {'base_weights': 'imagenet', 'units1': 512, 'unit